# 🔍 Agent 3 — Analyst
## Processes raw scraped data into structured competitive intelligence

**What this agent does:**
- Loads all raw JSON files from Agent 2
- Builds pricing comparison, feature matrix, messaging analysis
- Scores competitors on threat level
- Identifies gaps and opportunities
- Saves `data/analysed/full_analysis.json` for Agent 4

**SDK: Anthropic (Claude) — NOT OpenAI**

**Input ← Agent 2 | Output → Agent 4**

## 1. Install Dependencies

In [ ]:
%pip install anthropic python-dotenv --quiet

## 2. Setup

In [ ]:
import os, json
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv
import anthropic

load_dotenv()
print('ANTHROPIC_API_KEY loaded:', bool(os.getenv('ANTHROPIC_API_KEY')))

BASE_DIR      = Path('.')
DATA_RAW      = BASE_DIR / 'data' / 'raw'
DATA_ANALYSED = BASE_DIR / 'data' / 'analysed'
DATA_ANALYSED.mkdir(parents=True, exist_ok=True)

with open('input_schema.json') as f:
    INPUT = json.load(f)

client = anthropic.Anthropic()
print('Ready. Business:', INPUT['business']['name'])

## 3. Load Raw Data from Agent 2

In [ ]:
RAW_DATA = {}
for f in sorted(DATA_RAW.glob('*_raw.json')):
    try:
        d = json.loads(f.read_text())
        name = d.get('competitor_name', f.stem)
        RAW_DATA[name] = d
        status = d.get('scrape_status', 'unknown')
        sections = [s for s in ['pricing','features','homepage_headline'] if d.get(s)]
        print(f'  Loaded: {name} [{status}] — {len(sections)} key sections')
    except Exception as e:
        print(f'  Error loading {f.name}: {e}')

print(f'Total: {len(RAW_DATA)} competitors loaded')
if not RAW_DATA:
    print('WARNING: No raw data found. Run Agent 2 first.')

## 4. Define Tools

In [ ]:
# ── Tool functions ──────────────────────────────────────────────

def get_all_raw_data() -> str:
    """Returns all raw competitor scraped data plus our business context."""
    return json.dumps({
        'our_business': INPUT['business'],
        'competitors': RAW_DATA
    }, indent=2)


def build_pricing_comparison() -> str:
    """Normalizes and compares pricing across all competitors."""
    table = []
    for name, d in RAW_DATA.items():
        p = d.get('pricing', {})
        table.append({
            'competitor': name,
            'per_kg': p.get('per_kg', 'Unknown'),
            'express_surcharge': p.get('express_surcharge', 'Unknown'),
            'subscription_plan': p.get('subscription_plan', 'Unknown'),
            'minimum_order': p.get('minimum_order', 'Unknown'),
            'raw_pricing': p
        })
    our = INPUT['business']
    return json.dumps({
        'pricing_table': table,
        'our_pricing': {
            'model': our.get('usp', ''),
            'coverage': our.get('service_area', '')
        }
    }, indent=2)


def build_feature_matrix() -> str:
    """Creates a feature comparison matrix across all competitors."""
    features = [
        'Pickup & Delivery', 'Same-day Express', 'Dry Cleaning', 'Ironing',
        'Shoe Care', 'Subscription Plan', 'App Booking', 'Web Booking',
        'Eco-friendly Wash', 'Loyalty Program', 'GST Invoice', 'Coverage Map'
    ]
    matrix = {}
    for name, d in RAW_DATA.items():
        txt = ' '.join(str(f) for f in d.get('features', [])).lower()
        matrix[name] = {f: any(kw in txt for kw in f.lower().split()) for f in features}

    our_name = INPUT['business']['name']
    our_usp = INPUT['business'].get('usp', '').lower()
    matrix[our_name] = {
        f: any(kw in our_usp for kw in f.lower().split()) for f in features
    }
    return json.dumps({'feature_matrix': matrix, 'features': features}, indent=2)


def analyze_messaging_positioning() -> str:
    """Analyzes homepage messaging and positioning of each competitor."""
    return json.dumps({'messaging': [
        {
            'competitor': n,
            'headline': d.get('homepage_headline', 'N/A'),
            'usps': d.get('homepage_usp', []),
            'scrape_status': d.get('scrape_status', 'unknown')
        }
        for n, d in RAW_DATA.items()
    ]}, indent=2)


def analyze_content_and_jobs() -> str:
    """Analyzes blog content and hiring signals as growth indicators."""
    return json.dumps({'signals': [
        {
            'competitor': n,
            'blog_topics': d.get('blog_topics', [])[:5],
            'jobs_total': d.get('job_postings', {}).get('total', 0),
            'top_roles': d.get('job_postings', {}).get('top_roles', [])
        }
        for n, d in RAW_DATA.items()
    ]}, indent=2)


def identify_gaps_and_opportunities() -> str:
    """Identifies strategic gaps and opportunities vs competitors."""
    zones = INPUT['business'].get('zones', [])
    return json.dumps({
        'coverage_gaps': [f'{z} may be underserved by competitors' for z in zones[:3]],
        'service_gaps': [
            'Shoe care widely missing across competitors',
            'Curtain / sofa cleaning is a niche with low competition',
            'Subscription plans underutilized industry-wide'
        ],
        'messaging_gaps': [
            'No competitor prominently highlights eco-friendly wash',
            'Transparent per-kg pricing calculator missing from all sites'
        ],
        'opportunities': [
            'WhatsApp booking flow for non-app users',
            'Corporate tie-ups with IT parks and co-working spaces',
            'Eco-friendly certification as a marketing badge',
            'Loyalty punch card for repeat customers'
        ]
    }, indent=2)


def score_competitors() -> str:
    """Assigns a Competitive Threat Score (1-10) per competitor."""
    scores = []
    for name, d in RAW_DATA.items():
        feature_count = len(d.get('features', []))
        rating = d.get('reviews_summary', {}).get('rating', 3.5)
        job_count = d.get('job_postings', {}).get('total', 0)
        # Composite score: features/2 + rating*2 + jobs/10, averaged
        overall = round((min(10, feature_count / 2) + round(rating * 2, 1) + min(10, job_count / 10)) / 3, 1)
        scores.append({
            'competitor': name,
            'overall_threat_score': overall,
            'threat_level': 'high' if overall >= 7 else ('medium' if overall >= 5 else 'low'),
            'scrape_quality': d.get('scrape_status', 'unknown')
        })
    scores.sort(key=lambda x: x['overall_threat_score'], reverse=True)
    return json.dumps({'scores': scores}, indent=2)


def save_analysis_outputs(analysis_json: str) -> str:
    """Saves the complete structured analysis JSON for Agent 4."""
    p = DATA_ANALYSED / 'full_analysis.json'
    p.write_text(analysis_json)
    size = p.stat().st_size
    print(f'  Analysis saved: {p} ({size:,} bytes)')
    return str(p)


# ── Anthropic tool definitions ──────────────────────────────────
TOOLS = [
    {
        "name": "get_all_raw_data",
        "description": "Returns all raw competitor scraped data plus our business context.",
        "input_schema": {"type": "object", "properties": {}, "required": []}
    },
    {
        "name": "build_pricing_comparison",
        "description": "Builds a normalized pricing comparison table across all competitors.",
        "input_schema": {"type": "object", "properties": {}, "required": []}
    },
    {
        "name": "build_feature_matrix",
        "description": "Creates a feature comparison matrix (which competitor offers what).",
        "input_schema": {"type": "object", "properties": {}, "required": []}
    },
    {
        "name": "analyze_messaging_positioning",
        "description": "Analyzes competitor homepage messaging, headlines, and USPs.",
        "input_schema": {"type": "object", "properties": {}, "required": []}
    },
    {
        "name": "analyze_content_and_jobs",
        "description": "Analyzes blog topics and job postings as growth and strategy signals.",
        "input_schema": {"type": "object", "properties": {}, "required": []}
    },
    {
        "name": "identify_gaps_and_opportunities",
        "description": "Identifies strategic gaps in the market and opportunities for our business.",
        "input_schema": {"type": "object", "properties": {}, "required": []}
    },
    {
        "name": "score_competitors",
        "description": "Scores each competitor 1-10 on overall competitive threat level.",
        "input_schema": {"type": "object", "properties": {}, "required": []}
    },
    {
        "name": "save_analysis_outputs",
        "description": "Saves the complete structured analysis JSON to disk for Agent 4.",
        "input_schema": {
            "type": "object",
            "properties": {
                "analysis_json": {"type": "string", "description": "Complete analysis as a JSON string"}
            },
            "required": ["analysis_json"]
        }
    },
]

# ── Tool dispatcher ─────────────────────────────────────────────
TOOL_FNS = {
    "get_all_raw_data":               lambda **k: get_all_raw_data(),
    "build_pricing_comparison":        lambda **k: build_pricing_comparison(),
    "build_feature_matrix":            lambda **k: build_feature_matrix(),
    "analyze_messaging_positioning":   lambda **k: analyze_messaging_positioning(),
    "analyze_content_and_jobs":        lambda **k: analyze_content_and_jobs(),
    "identify_gaps_and_opportunities": lambda **k: identify_gaps_and_opportunities(),
    "score_competitors":               lambda **k: score_competitors(),
    "save_analysis_outputs":           lambda **k: save_analysis_outputs(**k),
}

print('Agent 3 tools ready:', [t['name'] for t in TOOLS])

## 5. Agentic Loop (Claude)

In [ ]:
def run_claude_agent(system: str, tools: list, tool_fns: dict, prompt: str,
                     model: str = 'claude-opus-4-8', max_tokens: int = 8192) -> str:
    """
    Runs a Claude agentic tool-use loop.
    Uses Opus for Agent 3 — deep reasoning needed for analysis.
    """
    messages = [{"role": "user", "content": prompt}]
    iteration = 0

    while True:
        iteration += 1
        print(f'  [loop {iteration}] calling Claude ({model})...')

        response = client.messages.create(
            model=model,
            max_tokens=max_tokens,
            system=system,
            tools=tools,
            messages=messages
        )

        if response.stop_reason == 'end_turn':
            return next((b.text for b in response.content if hasattr(b, 'text')), '')

        if response.stop_reason == 'tool_use':
            messages.append({"role": "assistant", "content": response.content})
            tool_results = []
            for block in response.content:
                if block.type == 'tool_use':
                    print(f'    → tool: {block.name}({list((block.input or {}).keys())})')
                    fn = tool_fns.get(block.name)
                    try:
                        result = fn(**(block.input or {})) if fn else f'Unknown tool: {block.name}'
                    except Exception as e:
                        result = f'Tool error: {e}'
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": str(result)
                    })
            messages.append({"role": "user", "content": tool_results})
        else:
            return f'Unexpected stop_reason: {response.stop_reason}'

## 6. Run Agent 3

In [ ]:
SYSTEM = f"""
You are Agent 3 — the Analyst in a competitive intelligence pipeline.
You are analysing competitors for: {INPUT['business']['name']} ({INPUT['business']['type']}).

WORKFLOW (follow in order):
1. get_all_raw_data()
2. build_pricing_comparison()
3. build_feature_matrix()
4. analyze_messaging_positioning()
5. analyze_content_and_jobs()
6. identify_gaps_and_opportunities()
7. score_competitors()
8. Compile a master analysis JSON with these keys:
   - analysis_date (ISO)
   - our_business (name, type, usp)
   - executive_summary (3 sentences — insight-rich, not generic)
   - pricing_analysis (from step 2)
   - feature_matrix (from step 3)
   - messaging_analysis (from step 4)
   - strategic_signals (from step 5)
   - gaps_and_opportunities (from step 6)
   - competitor_scores (from step 7)
   - top_3_recommendations (specific, actionable)
9. save_analysis_outputs(master_json_string)

RULES:
- Use only real scraped data — never invent facts
- If data is missing for a competitor, note it explicitly
- Be specific with numbers and evidence
"""

PROMPT = (
    'Analyse all competitor data. Build pricing comparison, feature matrix, '
    'messaging analysis, content signals, gaps, and threat scores. '
    'Compile a master JSON and save it via save_analysis_outputs.'
)

print('Running Agent 3 — Analyst (claude-opus-4-8)')
print('=' * 60)
t0 = datetime.now()

output = run_claude_agent(
    system=SYSTEM,
    tools=TOOLS,
    tool_fns=TOOL_FNS,
    prompt=PROMPT,
    model='claude-opus-4-8',
    max_tokens=8192
)

elapsed = (datetime.now() - t0).seconds
print(f'\nDone in {elapsed}s')
print(output[:600] if output else '(no text output)')

## 7. Verify Analysis Output

In [ ]:
af = DATA_ANALYSED / 'full_analysis.json'
if not af.exists():
    print('Analysis file not found — check agent output above')
else:
    a = json.loads(af.read_text())
    print(f'Analysis saved ({af.stat().st_size:,} bytes)')
    print(f'Keys: {list(a.keys())}')

    print('\nCompetitor Threat Scores:')
    for c in a.get('competitor_scores', []):
        score = c.get('overall_threat_score', 0)
        bar = '█' * int(score) + '░' * (10 - int(score))
        print(f'  {c["competitor"]:<20} {bar} {score}/10 [{c.get("threat_level","").upper()}]')

    print('\nTop 3 Recommendations:')
    for i, r in enumerate(a.get('top_3_recommendations', []), 1):
        print(f'  {i}. {r}')

    print('\nExecutive Summary:')
    print(' ', a.get('executive_summary', 'N/A'))

## ✅ Done — Next: open `04_agent4_report_writer.ipynb`